⊕ is symmetric difference.

M ← ∅

repeat
    P ← maximal set of vertex-disjoint shortest augmenting paths
    M ← M ⊕ (P1 ∪ P2 ∪ ... ∪ Pk)
until P = ∅

return M

In [1]:
"""
graph[u] = list of neighbors v
U = set/list of left partition vertices
V = set/list of right partition vertices
"""
from collections import deque

INF = float("inf")

def hopcroft_karp(graph, U, V):
    pair_u = {u: None for u in U}
    pair_v = {v: None for v in V}
    dist = {}

    def bfs():
        queue = deque()

        for u in U:
            if pair_u[u] is None:
                dist[u] = 0
                queue.append(u)
            else:
                dist[u] = INF

        dist[None] = INF

        while queue:
            u = queue.popleft()

            if dist[u] < dist[None]:
                for v in graph[u]:
                    if dist[pair_v[v]] == INF:
                        dist[pair_v[v]] = dist[u] + 1
                        queue.append(pair_v[v])

        return dist[None] != INF

    def dfs(u):
        if u is not None:
            for v in graph[u]:
                if dist[pair_v[v]] == dist[u] + 1:
                    if dfs(pair_v[v]):
                        pair_v[v] = u
                        pair_u[u] = v
                        return True

            dist[u] = INF
            return False

        return True

    matching = 0

    while bfs():
        for u in U:
            if pair_u[u] is None:
                if dfs(u):
                    matching += 1

    return matching, pair_u, pair_v

In [2]:
graph = {
    "A": [1, 2],
    "B": [1],
    "C": [2, 3],
}

U = ["A", "B", "C"]
V = [1, 2, 3]

matching, pair_u, pair_v = hopcroft_karp(graph, U, V)

print("Maximum matching size:", matching)
print("Matches from U:", pair_u)

Maximum matching size: 3
Matches from U: {'A': 2, 'B': 1, 'C': 3}


Let H be a directed graph (digraph).

The transpose of H, written Hᵀ, is defined as the graph obtained by
reversing the direction of every edge in H.

If H contains an edge:

    u → v

then Hᵀ contains the edge:

    v → u

Thus, the transpose operation reverses all edge directions.

Since H is acyclic, H contains no directed cycles. Reversing every edge
cannot create or destroy a cycle structure; it only changes the direction
in which paths are traversed. Therefore, Hᵀ is also acyclic.

Example:

H:
    A → B → C

Hᵀ:
    A ← B ← C

All edges are reversed, but there is still no directed cycle.

In [3]:
"""
Initial matching M from Figure 25.1(a):

Left side:
    L = {l1, l2, l3, l4, l5, l6, l7}

Right side:
    R = {r1, r2, r3, r4, r5, r6, r7, r8}

Initial matching:

M = {
    (l2, r2),
    (l3, r3),
    (l5, r7),
    (l7, r5)
}

Matched vertices in L:
    l2, l3, l5, l7

Therefore the FREE vertices in L are:

    l1, l4, l6

Matched vertices in R:
    r2, r3, r5, r7

Therefore the FREE vertices in R are:

    r1, r4, r6, r8

So the BFS does NOT start from all vertices l1–l7.
It starts only from the unmatched vertices:

    {l1, l4, l6}

That is exactly how Hopcroft–Karp constructs the BFS layers.

M = {
    (l2, r2),
    (l3, r3),
    (l5, r7),
    (l7, r5)
}

|M| = 4


--------------------------------------------------
STEP 1: Find all free (unmatched) vertices
--------------------------------------------------

Unmatched vertices in L:
    l1, l4, l6

Unmatched vertices in R:
    r1, r4, r6, r8


--------------------------------------------------
STEP 2: BFS phase
--------------------------------------------------

Hopcroft–Karp performs BFS from all free vertices in L
to find the shortest augmenting paths.

One shortest augmenting path is:

    l6 → r5 → l7 → r8

Explanation:
- (l6, r5) is NOT in M
- (r5, l7) corresponds to matched edge (l7, r5)
- (l7, r8) is NOT in M
- r8 is free

This is an augmenting path of odd length.


--------------------------------------------------
STEP 3: Augment the matching
--------------------------------------------------

Take the symmetric difference with the path.

Remove matched edges on the path:
    (l7, r5)

Add unmatched edges on the path:
    (l6, r5)
    (l7, r8)

New matching:

M1 = {
    (l2, r2),
    (l3, r3),
    (l5, r7),
    (l6, r5),
    (l7, r8)
}

|M1| = 5


--------------------------------------------------
STEP 4: Search for additional augmenting paths
--------------------------------------------------

Remaining unmatched vertices in L:
    l1, l4

Remaining unmatched vertices in R:
    r1, r4, r6

Running BFS again reveals another augmenting path:

    l1 → r2 → l2 → r1

Explanation:
- (l1, r2) is unmatched
- (l2, r2) is matched
- (l2, r1) is unmatched
- r1 is free


--------------------------------------------------
STEP 5: Augment again
--------------------------------------------------

Remove:
    (l2, r2)

Add:
    (l1, r2)
    (l2, r1)

New matching:

M2 = {
    (l1, r2),
    (l2, r1),
    (l3, r3),
    (l5, r7),
    (l6, r5),
    (l7, r8)
}

|M2| = 6


--------------------------------------------------
STEP 6: Check for more augmenting paths
--------------------------------------------------

The only unmatched left vertex is:
    l4

No augmenting path from l4 to a free right vertex exists.

Therefore, no more augmenting paths exist.


--------------------------------------------------
FINAL ANSWER
--------------------------------------------------

A maximum matching is:

M* = {
    (l1, r2),
    (l2, r1),
    (l3, r3),
    (l5, r7),
    (l6, r5),
    (l7, r8)
}

Maximum matching size:
    |M*| = 6
"""

'\nInitial matching M from Figure 25.1(a):\n\nLeft side:\n    L = {l1, l2, l3, l4, l5, l6, l7}\n\nRight side:\n    R = {r1, r2, r3, r4, r5, r6, r7, r8}\n\nInitial matching:\n\nM = {\n    (l2, r2),\n    (l3, r3),\n    (l5, r7),\n    (l7, r5)\n}\n\nMatched vertices in L:\n    l2, l3, l5, l7\n\nTherefore the FREE vertices in L are:\n\n    l1, l4, l6\n\nMatched vertices in R:\n    r2, r3, r5, r7\n\nTherefore the FREE vertices in R are:\n\n    r1, r4, r6, r8\n\nSo the BFS does NOT start from all vertices l1–l7.\nIt starts only from the unmatched vertices:\n\n    {l1, l4, l6}\n\nThat is exactly how Hopcroft–Karp constructs the BFS layers.\n\nM = {\n    (l2, r2),\n    (l3, r3),\n    (l5, r7),\n    (l7, r5)\n}\n\n|M| = 4\n\n\n--------------------------------------------------\nSTEP 1: Find all free (unmatched) vertices\n--------------------------------------------------\n\nUnmatched vertices in L:\n    l1, l4, l6\n\nUnmatched vertices in R:\n    r1, r4, r6, r8\n\n\n----------------------------

In the Hopcroft–Karp algorithm, after the BFS builds layers,
the graph H contains only edges that belong to shortest augmenting paths.

Suppose:
- Layer 0 contains the unmatched vertices in L.
- Layer q is the first layer containing an unmatched vertex in R.

Searching in Hᵀ from unmatched vertices in layer q back to layer 0
has an important advantage over searching forward in H.


Advantage of searching in Hᵀ
----------------------------

Searching backward in Hᵀ efficiently constructs a maximal set of
vertex-disjoint shortest augmenting paths.

Reason:

1. Start from successful endpoints
   - Every search begins at an unmatched vertex in R,
     which is already known to terminate a shortest augmenting path.
   - Thus every DFS is guaranteed to pursue only potentially useful paths.

2. Avoid unnecessary exploration
   - If searching forward from layer 0 in H,
     many paths may dead-end before reaching layer q.
   - Backward search avoids exploring paths that cannot end
     at a free vertex in R.

3. Easier to enforce vertex-disjointness
   - Once a path is chosen from layer q back to layer 0,
     its vertices can immediately be removed or marked.
   - This prevents overlap with future augmenting paths.

4. Guarantees maximal collection of shortest augmenting paths
   - The backward DFS naturally collects many disjoint shortest paths
     in one phase.
   - This is the key efficiency improvement of Hopcroft–Karp.

5. The graph is acyclic
   - H is a layered DAG.
   - Therefore Hᵀ is also acyclic.
   - Backward DFS proceeds cleanly from higher layers to lower layers
     without cycles.


Summary
-------

Searching in Hᵀ from unmatched vertices in layer q:
- starts only from valid endpoints,
- avoids dead-end searches,
- makes vertex-disjoint path construction easier,
- and efficiently finds a maximal set of shortest augmenting paths.

This is more efficient than searching forward from layer 0 in H,
where many explored paths may never reach a free vertex in R.

In [4]:
""" ========================================================================================================== """

' ========================================================================================================== '

Similarity between M-augmenting paths and augmenting paths in flow networks
--------------------------------------------------------------------------

Both concepts are used to improve a current solution incrementally.

1. Increase the current solution
   - In bipartite matching, an M-augmenting path increases the size
     of the matching by 1.
   - In flow networks, an augmenting path increases the total flow.

2. Alternate between usable and unusable edges
   - In matching, the path alternates between:
         unmatched edge, matched edge, unmatched edge, ...
   - In flow networks, the path alternates through edges with
     remaining residual capacity.

3. Start and end at “free capacity”
   - Matching:
         begins and ends at unmatched vertices.
   - Flow:
         begins at source s and ends at sink t where more flow can pass.

4. Used repeatedly until optimal
   - Hopcroft–Karp repeatedly finds augmenting paths until none exist.
   - Ford–Fulkerson repeatedly finds augmenting paths until none exist.

5. No augmenting path implies optimality
   - If no M-augmenting path exists, the matching is maximum.
   - If no augmenting path exists in the residual graph,
     the flow is maximum.


Differences between the two concepts
------------------------------------

1. Type of structure
   - Matching:
         works on bipartite graphs.
   - Flow:
         works on directed flow networks with capacities.

2. What is being augmented
   - Matching:
         edges are either matched or unmatched.
   - Flow:
         numerical flow values are increased.

3. Edge behavior
   - Matching:
         augmentation flips edges:
             matched ↔ unmatched
   - Flow:
         augmentation changes residual capacities.

4. Path constraints
   - Matching:
         paths must alternate between matched and unmatched edges.
   - Flow:
         paths only require positive residual capacity.

5. Amount added
   - Matching:
         each augmenting path increases matching size by exactly 1.
   - Flow:
         an augmenting path may increase flow by any amount equal
         to the bottleneck capacity.

6. Underlying representation
   - Matching:
         uses symmetric difference:
             M ← M ⊕ P
   - Flow:
         uses residual graphs and capacity updates.

In [5]:
"""
A = {l1, l2, l3}

- l1 connects to r1
- l2 connects to r1
- l3 connects to r1

N(A) = {r1}

|A| = 3
|N(A)| = 1

A perfect matching is impossible,
because 3 left vertices cannot all match to 1 right vertex.

perfect matching proves: |A| <= |N(A)|


Example: a 2-regular bipartite graph
L = {l1, l2, l3}
R = {r1, r2, r3}

Edges:
l1 -- r1
l1 -- r2

l2 -- r2
l2 -- r3

l3 -- r3
l3 -- r1

Every vertex has degree 2, so this is a 2-regular graph.

We use Hall's theorem.

Take any subset A ⊆ L.

Each vertex in A has degree d, so the total number of edges
leaving A equals:

    d|A|

All these edges go into N(A).

Since every vertex in N(A) has degree at most d,
the total number of edges entering N(A) is at most:
    d|N(A)|

Therefore:
    d|A| ≤ d|N(A)|

Dividing by d gives:
    |A| ≤ |N(A)|

Thus Hall's condition holds.

By Hall's theorem, G contains a perfect matching.
"""

"\nA = {l1, l2, l3}\n\n- l1 connects to r1\n- l2 connects to r1\n- l3 connects to r1\n\nN(A) = {r1}\n\n|A| = 3\n|N(A)| = 1\n\nA perfect matching is impossible,\nbecause 3 left vertices cannot all match to 1 right vertex.\n\nperfect matching proves: |A| <= |N(A)|\n\n\nExample: a 2-regular bipartite graph\nL = {l1, l2, l3}\nR = {r1, r2, r3}\n\nEdges:\nl1 -- r1\nl1 -- r2\n\nl2 -- r2\nl2 -- r3\n\nl3 -- r3\nl3 -- r1\n\nEvery vertex has degree 2, so this is a 2-regular graph.\n\nWe use Hall's theorem.\n\nTake any subset A ⊆ L.\n\nEach vertex in A has degree d, so the total number of edges\nleaving A equals:\n\n    d|A|\n\nAll these edges go into N(A).\n\nSince every vertex in N(A) has degree at most d,\nthe total number of edges entering N(A) is at most:\n    d|N(A)|\n\nTherefore:\n    d|A| ≤ d|N(A)|\n\nDividing by d gives:\n    |A| ≤ |N(A)|\n\nThus Hall's condition holds.\n\nBy Hall's theorem, G contains a perfect matching.\n"

In [6]:
from collections import deque


def gale_shapley(women_prefs, men_prefs):
    women = list(women_prefs.keys())
    men = list(men_prefs.keys())

    free_women = deque(women)

    woman_partner = {w: None for w in women}
    man_partner = {m: None for m in men}

    # Track next proposal index for each woman
    next_choice = {w: 0 for w in women}

    # smaller index = more preferred
    men_rank = {
        m: {w: rank for rank, w in enumerate(men_prefs[m])}
        for m in men
    }

    while free_women:
        w = free_women.popleft()

        # Next man on w's list
        m = women_prefs[w][next_choice[w]]
        next_choice[w] += 1

        # If m is free
        if man_partner[m] is None:
            woman_partner[w] = m
            man_partner[m] = w

        else:
            current_woman = man_partner[m]

            # Does m prefer new woman?
            if men_rank[m][w] < men_rank[m][current_woman]:

                # Break old engagement
                woman_partner[current_woman] = None
                free_women.append(current_woman)

                # New engagement
                woman_partner[w] = m
                man_partner[m] = w

            else:
                free_women.append(w)

    return woman_partner

In [25]:
women_prefs = {
    "Wanda": ["Brent", "Hank", "Oscar", "Davis"],
    "Emma": ["Davis", "Hank", "Oscar", "Brent"],
    "Lacey": ["Brent", "Davis", "Hank", "Oscar"],
    "Karen": ["Brent", "Hank", "Davis", "Oscar"]
}

men_prefs = {
    "Oscar": ["Wanda", "Karen", "Lacey", "Emma"],
    "Davis": ["Wanda", "Lacey", "Karen", "Emma"],
    "Brent": ["Lacey", "Karen", "Wanda", "Emma"],
    "Hank": ["Lacey", "Wanda", "Emma", "Karen"]
}

matching = gale_shapley(women_prefs, men_prefs)

print(matching)

{'Wanda': 'Hank', 'Emma': 'Oscar', 'Lacey': 'Brent', 'Karen': 'Davis'}


In [26]:
def greedy_bipartite_matching(L, adjacency):
    matching = set()
    matched_right = set()

    for l in L:
        for r in adjacency.get(l, []):
            if r not in matched_right:
                matching.add((l, r))
                matched_right.add(r)
                break

    return matching

In [27]:
L = ['A', 'B', 'C']

adjacency = {
    'A': [1, 2],
    'B': [1],
    'C': [2, 3]
}

result = greedy_bipartite_matching(L, adjacency)

print(result)

{('C', 2), ('A', 1)}


In [ ]:
def equality_graph(weights, hl, hr):
    n = len(weights)
    G = {i: [] for i in range(n)}

    for i in range(n):
        for j in range(n):
            if hl[i] + hr[j] == weights[i][j]:
                G[i].append(j)

    return G

In [ ]:
from collections import deque

def find_augmenting_path(G, match_r, n):
    parent_r = [None] * n
    visited_l = [False] * n
    visited_r = [False] * n

    q = deque()

    matched_left = set(match_r.values())

    # start from unmatched left vertices
    for l in range(n):
        if l not in matched_left:
            q.append(l)
            visited_l[l] = True

    while q:
        l = q.popleft()

        for r in G[l]:

            if visited_r[r]:
                continue

            visited_r[r] = True
            parent_r[r] = l

            # unmatched right vertex found
            if r not in match_r:
                return parent_r, r

            # continue alternating path
            next_l = match_r[r]

            if not visited_l[next_l]:
                visited_l[next_l] = True
                q.append(next_l)

    return None

In [ ]:
def augment_matching(match_r, parent_r, end_r):
    r = end_r

    while r is not None:
        l = parent_r[r]

        # old matched edge for l
        next_r = None

        for rr, ll in match_r.items():
            if ll == l:
                next_r = rr
                break

        # flip edge
        match_r[r] = l

        r = next_r

In [ ]:
def hungarian(weights):
    n = len(weights)
    hl = [max(row) for row in weights]
    hr = [0] * n

    match_r = {}

    while len(match_r) < n:
        G = equality_graph(weights, hl, hr)
        result = find_augmenting_path(G, match_r, n)

        if result:
            parent_r, end_r = result
            augment_matching(match_r, parent_r, end_r)

        else:
            slack = float('inf')

            for l in range(n):
                for r in range(n):

                    if r not in G[l]:
                        slack = min(
                            slack,
                            hl[l] + hr[r] - weights[l][r]
                        )

            for i in range(n):
                hl[i] -= slack

            for j in range(n):
                hr[j] += slack

    return match_r

In [ ]:
weights = [
    [8, 4, 5],
    [6, 7, 3],
    [5, 8, 1]
]

print(hungarian(weights))

In [ ]:
"""
| Term                        | Definition                                | Increases matching? |
| --------------------------- | ----------------------------------------- | ------------------- |
| Alternating / matching path | Edges alternate matched/unmatched         | Not necessarily     |
| Augmenting path             | Alternating path whose endpoints are free | Yes                 |

Every augmenting path is alternating, but not every alternating path is augmenting.

Hungarian algorithm alternates between:
Search for augmenting path
Get stuck
Create new equality edges using slack
Continue search
Augment matching
until perfect matching exists.


The minimum slack is the EXACT amount needed to create a new equality edge while preserving the invariant:
h(l) + h(r) ≥ w(l,r)

The labels are actually the "dual variables" of a linear program.

Equality edges correspond to:
tight constraints

Slack tells us:
how far a constraint is from becoming tight.

The algorithm carefully moves the dual variables until enough tight edges exist to build the optimal matching.
That's why Hungarian guarantees an optimal solution, not just a greedy one.

If ∣L∣ != ∣R∣, modify the assignment problem by
Adding dummy vertices to the smaller side until:
∣L∣ = ∣R∣

Add edges between dummy vertices and all vertices on the opposite side with appropriate weights:
usually 0 for minimization problems
or neutral/penalty values as appropriate.
Run the ordinary Hungarian algorithm on the resulting square bipartite graph.
Ignore matches involving dummy vertices when interpreting the final solution.
"""

In [ ]:
"""
EulerTour(G):
    choose any start vertex s
    stack = [s]
    tour = empty list

    while stack not empty:
        v = top(stack)

        if v has an unused incident edge (v,u):
            mark (v,u) used
            push u onto stack
        else:
            pop v from stack
            append v to tour

    reverse(tour)
    return tour
"""